In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xarray as xr

In [2]:
# Color Scheme for Risk Levels (consistent throughout all Charts)
COLORS = {
    "Low":      "#2ECC71",
    "Medium":   "#F39C12",
    "High":     "#E67E22",
    "Critical": "#E74C3C",
    "Unknown":  "#BDC3C7"
}

In [3]:
#load data
panel = pd.read_csv("panel_dataset.csv")
panel = panel.dropna(subset=["risk_score"])  # Dahab out fpr Visuals
print(f"{len(panel)} Spots geladen")
panel.head()

49 Spots geladen


,name,country,region,lat,lon,bleaching_freq_significant,bleaching_freq_severe,sst_trend_per_decade,score_bleaching,score_trend,risk_score,risk_level
0,Great Barrier Reef - Osprey Reef,Australia,Indo-Pacific,-13.8833,146.5667,10.0,6.0,0.294188,50.0,56.504320,52.6,High
1,Great Barrier Reef - Cod Hole,Australia,Indo-Pacific,-16.1167,145.9833,7.0,2.0,0.255187,35.0,47.361285,39.9,Medium
2,Coral Sea - Holmes Reef,Australia,Indo-Pacific,-16.4833,147.8667,10.0,4.0,0.288779,50.0,55.236211,52.1,High
3,Ribbon Reefs - No. 10,Australia,Indo-Pacific,-15.0833,145.7500,5.0,3.0,0.262589,25.0,49.096538,34.6,Medium
4,Raja Ampat - Misool,Indonesia,Indo-Pacific,-2.0833,130.1667,0.0,0.0,0.193966,0.0,33.009020,13.2,Low


In [4]:
# world map
fig_map = px.scatter_geo(
    panel,
    lat="lat",
    lon="lon",
    color="risk_level",
    color_discrete_map=COLORS,
    hover_name="name",
    hover_data={
        "country": True,
        "risk_score": True,
        "risk_level": True,
        "bleaching_freq_significant": True,
        "sst_trend_per_decade": ":.3f",
        "lat": False,
        "lon": False
    },
    size="risk_score",
    size_max=18,
    projection="natural earth",
    title="Ocean Eyes — Dive Spot Climate Risk Index",
    category_orders={"risk_level": ["Low", "Medium", "High", "Critical"]}
)

fig_map.update_layout(
    paper_bgcolor="#0a1628",
    plot_bgcolor="#0a1628",
    font_color="white",
    geo=dict(
        bgcolor="#0a1628",
        landcolor="#1a2f4a",
        oceancolor="#0d2137",
        showocean=True,
        showland=True,
        showcoastlines=True,
        coastlinecolor="#2a4a6a",
        showframe=False,
    ),
    legend_title_text="Risk Level",
    margin=dict(l=0, r=0, t=50, b=0),
    height=550
)

fig_map.write_html("world_map.html")
fig_map.show()
print("world_map.html saved")

world_map.html saved


In [5]:
# ranking top 20 spots most at risk
top20 = panel.sort_values("risk_score", ascending=True).tail(20)

fig_rank = go.Figure(go.Bar(
    x=top20["risk_score"],
    y=top20["name"],
    orientation="h",
    marker_color=[COLORS[lvl] for lvl in top20["risk_level"]],
    text=top20["risk_level"],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Risk Score: %{x}<extra></extra>"
))

fig_rank.update_layout(
    title="Top 20 Most At-Risk Dive Spots",
    xaxis_title="Composite Risk Score (0–100)",
    yaxis_title="",
    paper_bgcolor="#0a1628",
    plot_bgcolor="#0a1628",
    font_color="white",
    xaxis=dict(range=[0, 105], gridcolor="#1a2f4a"),
    yaxis=dict(gridcolor="#1a2f4a"),
    height=600,
    margin=dict(l=200, r=100, t=50, b=50)
)

fig_rank.write_html("top20_risk_ranking.html")
fig_rank.show()
print("top20_risk_ranking.html saved")

top20_risk_ranking.html saved


In [ ]:
# Scatter (Bleaching Freq vs SST Trend)
fig_scatter = px.scatter(
    panel,
    x="sst_trend_per_decade",
    y="bleaching_freq_significant",
    color="risk_level",
    color_discrete_map=COLORS,
    hover_name="name",
    hover_data={"risk_score": True, "country": True},
    size="risk_score",
    size_max=20,
    text="name",
    title="Bleaching Frequency vs. SST Warming Trend",
    labels={
        "sst_trend_per_decade": "SST Trend (°C / decade)",
        "bleaching_freq_significant": "Bleaching Events (DHW ≥ 4, since 1985)"
    },
    category_orders={"risk_level": ["Low", "Medium", "High", "Critical"]}
)

fig_scatter.update_traces(textposition="top center", textfont_size=8)
fig_scatter.update_layout(
    paper_bgcolor="#0a1628",
    plot_bgcolor="#0a1628",
    font_color="white",
    xaxis=dict(gridcolor="#1a2f4a"),
    yaxis=dict(gridcolor="#1a2f4a"),
    height=550
)

fig_scatter.write_html("scatter.html")
fig_scatter.show()
print("scatter.html saved")

scatter.html gespeichert


In [7]:
def spot_detail(spot_name):
    spot = panel[panel["name"] == spot_name].iloc[0]
    
    # Gauge Chart für Risk Score
    fig = go.Figure()
    
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=spot["risk_score"],
        title={"text": f"{spot['name']}<br><sub>{spot['country']} · {spot['region']}</sub>"},
        gauge={
            "axis": {"range": [0, 100]},
            "bar": {"color": COLORS[spot["risk_level"]]},
            "steps": [
                {"range": [0, 25],  "color": "#1a3a2a"},
                {"range": [25, 50], "color": "#3a3010"},
                {"range": [50, 75], "color": "#3a2010"},
                {"range": [75, 100],"color": "#3a1010"},
            ],
            "threshold": {
                "line": {"color": "white", "width": 2},
                "thickness": 0.75,
                "value": spot["risk_score"]
            }
        },
        number={"suffix": " / 100", "font": {"color": COLORS[spot["risk_level"]]}}
    ))
    
    # Metriken als Annotations
    fig.add_annotation(x=0.25, y=0.15, xref="paper", yref="paper",
        text=f"<b>Bleaching Events</b><br>{int(spot['bleaching_freq_significant'])} since 1985",
        showarrow=False, font=dict(size=13, color="white"),
        bgcolor="#1a2f4a", borderpad=8)
    
    fig.add_annotation(x=0.75, y=0.15, xref="paper", yref="paper",
        text=f"<b>SST Trend</b><br>+{spot['sst_trend_per_decade']:.3f}°C / decade",
        showarrow=False, font=dict(size=13, color="white"),
        bgcolor="#1a2f4a", borderpad=8)
    
    fig.add_annotation(x=0.5, y=0.02, xref="paper", yref="paper",
        text=f"Risk Level: <b>{spot['risk_level']}</b>  ·  Data: NOAA Coral Reef Watch 1985–2025",
        showarrow=False, font=dict(size=11, color="#BDC3C7"))
    
    fig.update_layout(
        paper_bgcolor="#0a1628",
        font_color="white",
        height=420,
        margin=dict(l=30, r=30, t=80, b=80)
    )
    
    return fig

# Beispiel — kannst du mit jedem Spot testen
fig_detail = spot_detail("Palau - Blue Corner")
fig_detail.show()